# Gemma 4 Developer Agent: Replay Architecture Submission

Autonomous developer agent built on **Google Gemma 4 31B QAT** (`gemma-4-31b-it-qat-w4a16-ct`), featuring:
- **Single agent, core tools only** (V3): prose instructions, targeted reads, per-task budget in `eval_config.yaml`.
- **Disciplined Replay Cycle**: `/tmp/repro.py` failure reproduction -> atomic `edit_file` patch -> targeted pytest feedback with output tailing (`tail -n 40`) to mitigate the 5,000-character truncation limit.
- **No premature timeouts**: Unrestricted per-task execution by dropping the restrictive 1-minute `eval_config.yaml`.
- **Optimized Sampling**: 4,096 thinking budget with `include_thoughts: false` to keep thought overhead outside conversation context.

In [ ]:
import os
import sys
import shutil
import zipfile
import subprocess
from pathlib import Path
import yaml
import re

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./working')
WORKING.mkdir(parents=True, exist_ok=True)
BUNDLE = WORKING / 'submission_bundle'
ZIP_PATH = WORKING / 'submission.zip'
print(f'Target zip: {ZIP_PATH}')


In [ ]:
BUNDLE_FILES = {'agent.yaml': 'name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Python developer agent that fixes one issue with a small, verified source patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: false\n', 'prompts/system.md': 'You are a software engineer fixing one issue in the Python repository at /workspace. Nobody will answer questions: work with your tools until the fix is in place, then submit it.\n\nHow your work is graded: hidden tests run on your source changes in a fresh checkout. A focused fix of the real cause is what counts. Test files are reset before grading, so never edit existing tests, pytest.ini or conftest.py. Follow the issue literally: keep the exact function, class, parameter, error message and exception names it asks for.\n\nYour tools are run_command (a shell in /workspace, offline, 300-second timeout, output cut after 5,000 characters), read_file (at most 150 lines per call), edit_file, write_file, get_status and submit_patch. Do not use edit_file: its arguments are often lost on this system. Use only these tool names. Shell programs such as grep, sed, git or python3 are run through run_command, never called as tools.\n\nWork in this order.\n\nFirst, locate the code. Take the identifiers from the issue (names, messages, options, paths) and search for them with git grep -n through run_command, limited to Python files and piped to head -n 25. Before reading a long file, find the right line range with grep -n on "def " and "class ", then inspect that range using read_file or using sed -n \'START,END p\' filepath through run_command. Do not read the same range twice, and never run the same command twice: if a search gives nothing new, change the search terms or read the code instead. Follow a call to its definition when the bug is in the callee.\n\nSecond, if it helps, reproduce the problem: write a short script under /tmp with a shell heredoc through run_command and run it with python3. Skip this when the issue is already clear from the code.\n\nThird, edit, always with a short Python script run through run_command. Write it as a heredoc: python3 - <<\'EOF\', then a script that imports pathlib, reads the file with p = pathlib.Path(path) and s = p.read_text(), sets old to the exact lines to replace (copied from what you read, same indentation, in a triple-quoted string) and new to the replacement, checks assert s.count(old) == 1, writes p.write_text(s.replace(old, new, 1)), and prints "edited"; then a line with EOF. Keep old short and unique: a few consecutive lines. If the assert fails, inspect the lines again and retry with a corrected old. Change as little as needed and keep public interfaces compatible. Prefer several small edits to one large one.\n\nFourth, verify. Check the edited file compiles with python3 -m py_compile filepath. Run only the most relevant test file, with pytest -q -x and a -k filter when useful, and pipe the output to tail -n 40. Never run the whole test suite. If a test fails, read the end of the trace and fix the cause.\n\nFifth, submit promptly. As soon as your fix compiles and the targeted test passes (or confirms the fix), do not keep searching or experimenting. Make sure no scratch file was created inside /workspace (check git status --short through run_command, and remove any temporary file in /tmp). Then call submit_patch as your final action. A focused, compiling fix submitted is always better than no patch.\n'}

shutil.rmtree(BUNDLE, ignore_errors=True)
for rel_path, content in BUNDLE_FILES.items():
    p = BUNDLE / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')

print(f'Extracted {len(BUNDLE_FILES)} files into {BUNDLE}:')
for p in sorted(BUNDLE.rglob('*')):
    if p.is_file():
        print(f'  {p.stat().st_size:>6} B  {p.relative_to(BUNDLE)}')


In [ ]:
# Standalone Rule Validation
ROOT_CONFIGS = ['agent.yaml', 'agent.yml', 'root_agent.yaml', 'root_agent.yml']
ALLOWED_SUFFIXES = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
yaml.SafeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))

def validate(bundle: Path) -> None:
    files = [p for p in bundle.rglob('*') if p.is_file()]
    roots = [name for name in ROOT_CONFIGS if (bundle / name).is_file()]
    assert len(roots) == 1, f'Need exactly one root config, found {roots}'
    assert not any(p.is_symlink() for p in bundle.rglob('*')), 'Symlinks are forbidden'
    assert all(p.suffix in ALLOWED_SUFFIXES for p in files), 'Disallowed file suffixes found'
    total_bytes = sum(p.stat().st_size for p in files)
    assert total_bytes < 3 * (1024**3), 'Unpacked bundle exceeds 3 GiB'
    
    models = set()
    for config in (p for p in files if p.suffix in ('.yaml', '.yml')):
        text = config.read_text(encoding='utf-8')
        data = yaml.safe_load(text) or {}
        for target in re.findall(r'!include\s+(\S+)', text):
            target_path = (config.parent / target).resolve()
            assert target_path.is_file(), f'{config.name}: missing include {target}'
        if 'model' in data:
            models.add(data['model'].split('/')[-1])
            
    assert len(models) == 1, f'One base model per submission, found {models}'
    assert list(models)[0] == 'gemma-4-31b-it-qat-w4a16-ct', f'Model must be gemma-4-31b-it-qat-w4a16-ct, got {models}'
    print(f'SUCCESS: Validated {roots[0]}, {len(files)} files, model: {list(models)[0]}, size: {total_bytes} bytes.')

validate(BUNDLE)


In [ ]:
# Optional Harness Validation if wheelhouse is attached
HARNESS_WHEELS = ('google_adk-', 'google_genai-', 'adk_submission-', 'adk_eval_core-', 'swegemma-')
MODEL = 'gemma-4-31b-it-qat-w4a16-ct'

def install_harness():
    found = sorted(Path('/kaggle/input').glob('**/adk_submission-*.whl'))
    if found:
        wheelhouse = found[0].parent
        wheels = [str(w) for w in sorted(wheelhouse.glob('*.whl')) if w.name.startswith(HARNESS_WHEELS)]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels], check=True)

try:
    try:
        import adk_submission
    except ImportError:
        install_harness()
    from adk_submission import ModelRegistry, compile_submission, validate_directory
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import build_submission_limits
    from swegemma.models.discovery import validate_single_declared_model
    
    def run_command(command: str) -> str: ...
    def read_file(filepath: str, start_line: int | None = None, end_line: int | None = None) -> str: ...
    def write_file(filepath: str, content: str) -> str: ...
    def edit_file(filepath: str, old_string: str, new_string: str, allow_multiple: bool = False) -> str: ...
    def submit_patch() -> str: ...
    def get_status() -> str: ...
    def get_code_neighbors(node: str, edge_type: str | None = None, max_neighbors: int = 50) -> str: ...
    def search_similar_code(query: str, k: int = 10) -> str: ...
    def get_code_subgraph(nodes: list[str]) -> str: ...
    
    limits, constraints = build_submission_limits()
    models = ModelRegistry()
    models.register(MODEL, LiteLlm(model=f'openai/{MODEL}', api_base='http://127.0.0.1:9/v1', api_key='EMPTY'))
    tools = {f.__name__: f for f in [run_command, read_file, write_file, edit_file, submit_patch, get_status,
                                     get_code_neighbors, search_similar_code, get_code_subgraph]}
    validate_directory(BUNDLE, limits)
    print('validate_directory: OK')
    print('validate_single_declared_model:', validate_single_declared_model(BUNDLE))
    agent = compile_submission(BUNDLE, tools, models, limits=limits, generation_constraints=constraints)
    print(f'compile_submission: OK, root agent: {agent.name!r}')
except Exception as e:
    print(f'Official harness check skipped or unavailable ({e}); offline validation confirmed.')


In [ ]:
# Build Clean submission.zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in BUNDLE.rglob('*') if p.is_file()):
        archive.write(path, path.relative_to(BUNDLE).as_posix())

with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()

assert 'agent.yaml' in names, 'agent.yaml must be at the root of the archive'
assert not any(name.endswith('/') for name in names), 'Zip archive must contain files only'
shutil.rmtree(BUNDLE, ignore_errors=True)
print(f'✅ Successfully built {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)')
print('Archive entries:')
for n in names:
    print(f'  - {n}')
